# 02 — Prototypage des modèles d'uplift

**Hypothèses**
- **H1** : un modèle d'uplift classe mieux les clients *persuadables* qu'un modèle de réponse classique.
- **H2** : parmi S/T/X-learner, le meilleur dépend des données → on le choisit sur **validation**.
- **H3** : les hyperparamètres comptent peu face au bruit ; on privilégie des modèles simples et stables.

Le code de ce notebook est ensuite industrialisé dans `src/uplift/models.py` et `train.py`.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")            # pas d'installation nécessaire
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import load_config
cfg = load_config()
ARMS, CTRL = cfg["arms"], cfg["control_label"]
pd.set_option("display.float_format", lambda v: f"{v:.5f}")
from data import load_raw, prepare, split, FEATURES
from features import build_preprocessor
from evaluation import qini_curve, qini_coefficient, uplift_at_k, uplift_by_decile, plot_qini
from models import MultiArmUplift, TLearner, ResponseModel, _PARAMS
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

raw, source = load_raw(cfg); df = prepare(raw, cfg)
train, test = split(df, cfg["data"]["test_size"], cfg["seed"])
fit, val = split(train, cfg["data"]["val_size"], cfg["seed"])       # le test n'est PAS touché ici
pre = build_preprocessor().fit(fit[FEATURES])
Xf, Xv = pre.transform(fit[FEATURES]), pre.transform(val[FEATURES])
print(source, len(fit), len(val), len(test))

hillstrom 35840 8960 19200


## 1. Prototype « from scratch » : T-learner logistique sur un bras

In [2]:
arm = ARMS[0]
mf = (fit.treatment.isin([arm, CTRL])).to_numpy(); mv = (val.treatment.isin([arm, CTRL])).to_numpy()
tf = (fit.treatment[mf] == arm).to_numpy(); yf = fit.y[mf].to_numpy()
tv = (val.treatment[mv] == arm).astype(int).to_numpy(); yv = val.y[mv].to_numpy()

m1 = LogisticRegression(max_iter=1000).fit(Xf[mf][tf], yf[tf])
m0 = LogisticRegression(max_iter=1000).fit(Xf[mf][~tf], yf[~tf])
s_logit = m1.predict_proba(Xv[mv])[:, 1] - m0.predict_proba(Xv[mv])[:, 1]
print("Qini (T-learner logistique) :", round(qini_coefficient(yv, tv, s_logit), 6))
print("Qini (score aléatoire)       :", round(qini_coefficient(yv, tv, np.random.default_rng(0).random(len(yv))), 6))

Qini (T-learner logistique) : 0.000405
Qini (score aléatoire)       : -0.000199


**Conclusion** : le modèle bat très légèrement le hasard (Qini = +0,000405 contre −0,000199 pour un score aléatoire), mais l'écart est **minuscule et probablement dans le bruit**. À ce stade, rien ne prouve que le ciblage apporte quelque chose.

### Lecture des chiffres
- **Qini du T-learner : 0,000405**, soit environ **+0,04 point de conversion incrémentale** par client, en moyenne, par rapport à un ciblage au hasard.
- **Qini du score aléatoire : −0,000199**. Un score aléatoire devrait donner ≈ 0 : cet écart à 0 mesure directement **l'ordre de grandeur du bruit** sur ce jeu de validation.
- L'écart entre les deux (≈ 0,0006) est du **même ordre que le bruit**. Avec un seul tirage aléatoire, on ne peut pas conclure.

### Mise en perspective
- L'ATE de Mens est de **+0,68 pt**, contre **+0,04 pt** gagné par le ciblage : l'essentiel du gain vient de **l'email lui-même**, pas du choix des destinataires.
- Cohérent avec la section 4 : effet quasi homogène, donc peu de marge pour un ciblage.
- Un signe positif est tout de même encourageant pour un modèle simple (régression logistique, un seul bras, pas de réglage).

### Limites
- **Un seul chiffre, sans incertitude** : il faut un **IC par bootstrap** du Qini, et comparer à la **distribution** du Qini aléatoire (par exemple 200 tirages), pas à un seul tirage.
- **Jeu de validation petit** et ~1 % de conversion : très peu de conversions par tranche, donc courbe très instable.
- **Un seul bras** : le prototype porte sur `ARMS[0]` ; à répéter sur Womens.
- **Modèle linéaire** : peut manquer des interactions entre variables.
- Le Qini seul ne dit pas **où** le modèle gagne : à compléter par la **courbe de Qini**, l'**uplift@k** et les **déciles**.

### Suite
- Bootstrap du Qini et du score aléatoire (IC, p-value empirique).
- Comparer avec S-learner, T-learner gradient boosting et un ciblage « tout le monde ».
- Reproduire pour l'autre bras, puis passer à `MultiArmUplift`.

## 2. H1 — uplift vs modèle de réponse, et H2 — S/T/X-learner (validation)

In [3]:
rows, curves = [], {}
for name in ["response_model", "s_learner", "t_learner", "x_learner"]:
    m = MultiArmUplift(name, ARMS, CTRL, cfg["seed"]).fit(Xf, fit.treatment, fit.y)
    U = m.predict_uplift(Xv)
    r = {"model": name}
    for a in ARMS:
        k = val.treatment.isin([a, CTRL]).to_numpy()
        y, t, s = val.y.to_numpy()[k], (val.treatment.to_numpy()[k] == a).astype(int), U[a].to_numpy()[k]
        r[f"qini_{a.split()[0]}"] = qini_coefficient(y, t, s); r[f"upl@30_{a.split()[0]}"] = uplift_at_k(y, t, s, .3)
        if a == arm: curves[name] = qini_curve(y, t, s)
    rows.append(r)
res = pd.DataFrame(rows).set_index("model"); res["mean_qini"] = res.filter(like="qini_").mean(axis=1)
display(res)
plot_qini(curves, f"Validation — {arm}"); plt.show()

,qini_Mens,upl@30_Mens,qini_Womens,upl@30_Womens,mean_qini
model,,,,,
response_model,0.00047,0.01325,0.00033,0.00559,0.00040
s_learner,0.00034,0.00835,-0.00018,-0.00238,0.00008
t_learner,0.00055,0.00997,0.00006,0.00569,0.00031
x_learner,0.00003,0.00699,-0.00001,0.00362,0.00001


**Conclusion** : sur la validation, **aucun learner d'uplift ne bat clairement le simple modèle de réponse**, et les écarts entre learners sont **du même ordre que le bruit**. H1 n'est pas soutenue, H2 donne un léger avantage au T-learner, sans preuve solide.

### H1 : l'uplift fait-il mieux qu'un modèle de réponse ?
- Le **modèle de réponse** obtient le meilleur Qini moyen (≈ 0,0004), devant le T-learner (≈ 0,0003).
- Les learners d'uplift n'apportent donc **pas de gain** par rapport à un classifieur qui prédit simplement qui convertit.
- Hypothèse cohérente avec la section 4 : l'effet est quasi homogène en **relatif**. Si l'email multiplie la conversion d'un facteur à peu près constant, les clients à forte probabilité de base ont aussi le plus grand gain **absolu**, et un classifieur de réponse capte déjà ce signal.
- L'uplift@30 du modèle de réponse (≈ +1,3 pt pour Mens, ≈ +0,56 pt pour Womens) est environ **2 fois l'ATE** (+0,68 et +0,31 pt) : cibler les 30 % les plus probables semble utile, mais ce n'est pas propre à l'uplift modeling.

### H2 : S, T ou X-learner ?
- **T-learner** : meilleur des learners d'uplift (Qini moyen ≈ 0,0003), positif sur les deux bras.
- **S-learner** : faible (≈ 0,0001) et **négatif sur Womens** (uplift@30 ≈ −0,24 pt, donc pire que le hasard). Cohérent avec le biais attendu : avec des modèles régularisés et un effet faible, la colonne traitement est quasi ignorée et l'uplift est écrasé vers 0.
- **X-learner** : Qini ≈ 0 sur les deux bras, soit aucun gain mesurable. Son uplift@30 sur Mens (≈ +0,70 pt) est quasi égal à l'ATE (+0,68 pt) : aucun avantage à cibler.

### Mise en perspective
- Les ordres de grandeur sont **minuscules** : un Qini de 0,0004 correspond à ≈ +0,04 pt de conversion incrémentale par client, contre +0,68 pt pour l'ATE de Mens. L'essentiel du gain vient de **l'email**, pas du **ciblage**.
- Le prototype précédent donnait un Qini aléatoire de −0,0002 : le bruit est de l'ordre de **±0,0002**. Les écarts entre modèles (0,0001 à 0,0004) sont donc **dans le bruit**.
- Le classement observé (modèle de réponse ≈ T > S > X) est **plausible** mais non démontré.

### Limites
- **Aucun intervalle d'incertitude** : il faut un bootstrap du Qini et de l'uplift@30, avec la distribution du Qini d'un score aléatoire en référence.
- **Un seul découpage et une seule graine** : répéter sur plusieurs seeds ou en validation croisée.
- **Validation utilisée pour choisir** : le meilleur modèle est biaisé à la hausse, la comparaison finale doit se faire sur le **test**.
- **Peu de conversions par bras** (quelques centaines au total) : courbes et uplift@30 très instables.
- Le Qini moyen mélange deux bras de nature différente : à lire **bras par bras**.

### Décision proposée
- Retenir en priorité le **modèle de réponse** (plus simple, aussi performant) et le **T-learner** comme candidat uplift.
- Abandonner le S-learner (négatif sur Womens) et le X-learner (aucun gain).
- Confirmer par bootstrap puis sur le jeu de test, et décider du ciblage selon le **coût d'envoi** : si le coût est faible, envoyer à tous reste défendable.

**Lecture** : un Qini > 0 signifie mieux que le hasard ; comparer `response_model` aux learners causaux teste H1.
Avec un tel bruit, des écarts faibles entre learners ne sont pas concluants → voir le §3.

## 3. H3 — sensibilité aux hyperparamètres (3 graines)

In [4]:
def t_learner_qini(depth, leaf, seed):
    p = dict(_PARAMS, max_depth=depth, min_samples_leaf=leaf)
    a1 = HistGradientBoostingClassifier(random_state=seed, **p).fit(Xf[mf][tf], yf[tf])
    a0 = HistGradientBoostingClassifier(random_state=seed, **p).fit(Xf[mf][~tf], yf[~tf])
    return qini_coefficient(yv, tv, a1.predict_proba(Xv[mv])[:, 1] - a0.predict_proba(Xv[mv])[:, 1])

grid = []
for depth in [2, 3, 4]:
    for leaf in [50, 100, 300]:
        q = [t_learner_qini(depth, leaf, s) for s in range(3)]
        grid.append(dict(max_depth=depth, min_samples_leaf=leaf, qini_mean=np.mean(q), qini_std=np.std(q)))
grid = pd.DataFrame(grid); display(grid.pivot(index="max_depth", columns="min_samples_leaf", values="qini_mean"))
print("écart-type moyen entre graines :", round(grid.qini_std.mean(), 6),
      "| amplitude entre configs :", round(grid.qini_mean.max() - grid.qini_mean.min(), 6))

min_samples_leaf,50,100,300
max_depth,,,
2,0.00056,0.00059,0.00052
3,0.00039,0.00045,0.00036
4,0.00028,0.00020,0.00033


écart-type moyen entre graines : 0.000122 | amplitude entre configs : 0.000392


**Conclusion** : le Qini est **faible dans toutes les configurations** (≈ 0,0002 à 0,0006), et **les modèles les plus simples (`max_depth = 2`) sont les meilleurs**. La profondeur a un effet visible ; `min_samples_leaf` n'a pas d'effet distinguable du bruit. Le choix exact des hyperparamètres reste fragile.

### Lecture de la grille
- **`max_depth`** : le Qini moyen **décroît avec la profondeur** : ≈ 0,00055 (profondeur 2), ≈ 0,00040 (profondeur 3), ≈ 0,00027 (profondeur 4). Plus le modèle est flexible, plus il ajuste du bruit, ce qui confirme le choix de modèles régularisés pour un événement rare (~1 %).
- **`min_samples_leaf`** : à profondeur fixée, les écarts entre 50, 100 et 300 sont de l'ordre de 0,00007 à 0,00013, soit **le niveau du bruit entre graines**. Aucune tendance nette.
- **Meilleure configuration** : `max_depth = 2`, `min_samples_leaf = 100` (≈ 0,00059), mais elle est **indiscernable** des deux autres configurations de profondeur 2.

### Bruit contre signal
- **Écart-type entre graines : ≈ 0,00012** : la variabilité due à la seule graine.
- **Amplitude entre configurations : ≈ 0,00039**, soit environ **3 fois** le bruit entre graines. Il existe donc un effet réel des hyperparamètres, surtout porté par la profondeur.
- Avec seulement 3 graines, l'erreur standard d'un Qini moyen reste de l'ordre de 0,00007 : les écarts **à l'intérieur d'une même profondeur** ne sont pas fiables, l'écart **entre profondeur 2 et 4** (≈ 0,0003) l'est davantage.
- Rappel : un score aléatoire donne un Qini d'environ ±0,0002. Même la meilleure configuration n'est qu'à **quelques fois** ce niveau de bruit.

### Ce que ça implique
- **Garder des arbres très peu profonds** (profondeur 2, éventuellement 3) et `min_samples_leaf` autour de 100 : choix raisonnable, sans surinterpréter les différences fines.
- Inutile de pousser le réglage : le gain potentiel (≈ 0,0003 de Qini) est minuscule face à l'effet de l'email lui-même (ATE de +0,68 pt pour Mens).
- Cohérent avec H1 et H2 : le signal d'hétérogénéité est faible, donc les modèles complexes n'ont rien de plus à capter.

### Limites
- **3 graines seulement** : trop peu pour estimer finement la variance. Passer à 10 graines ou à une validation croisée répétée.
- **Un seul découpage de validation et un seul bras** (`ARMS[0]`, Mens) : à reproduire sur Womens.
- **Sélection sur la validation** : la meilleure configuration est biaisée à la hausse. La performance finale doit être mesurée une seule fois sur le **test**.
- **Grille réduite** (9 configurations) : `learning_rate`, `max_iter` et `l2_regularization` restent fixés.
- Le Qini n'est pas accompagné d'un **IC** : à ajouter par bootstrap.

### Décision proposée
- Fixer `max_depth = 2` (ou 3) et `min_samples_leaf = 100` pour la suite.
- Ne pas investir davantage dans le réglage : confirmer sur le test, puis décider du ciblage selon le **coût d'envoi**.

## 4. Calibration par déciles (le classement est-il utile ?)

In [5]:
m = MultiArmUplift("t_learner", ARMS, CTRL, cfg["seed"]).fit(Xf, fit.treatment, fit.y)
U = m.predict_uplift(Xv)
k = val.treatment.isin([arm, CTRL]).to_numpy()
dec = uplift_by_decile(val.y.to_numpy()[k], (val.treatment.to_numpy()[k] == arm), U[arm].to_numpy()[k], bins=5)
display(dec)
dec.plot(x="decile", y=["mean_pred_uplift", "observed_uplift"], kind="bar", title="Uplift prédit vs observé par quintile"); plt.show()

,decile,n,mean_pred_uplift,rate_treated,rate_control,observed_uplift
0,1,1194,0.01005,0.01709,0.00328,0.01381
1,2,1193,0.00759,0.00812,0.00347,0.00465
2,3,1193,0.00665,0.01880,0.00329,0.01551
3,4,1193,0.00575,0.01165,0.01014,0.00151
4,5,1193,0.00396,0.00671,0.00838,-0.00166


**Conclusion** : le classement est **globalement dans le bon sens** (les quintiles du haut réagissent plus que ceux du bas) et **bien calibré en moyenne**, mais il n'est **pas monotone** et chaque quintile est très bruité. C'est un signe encourageant, pas une preuve d'un ciblage utile.

### Lecture du tableau
- Le découpage est en **5 quintiles** (`bins=5`), la colonne s'appelle `decile` par construction de la fonction. Chaque quintile contient ≈ 1 190 clients (traités et contrôle réunis).
- **Uplift prédit** : décroît régulièrement de **+1,0 pt** (quintile 1) à **+0,4 pt** (quintile 5). Le modèle classe donc bien selon son propre score.
- **Uplift observé** : +1,38 / +0,47 / +1,55 / +0,15 / −0,17 pt. Les quintiles 1 à 3 sont positifs, les quintiles 4 et 5 sont proches de 0 et même légèrement négatifs.
- **Calibration moyenne** : la moyenne de l'uplift prédit (≈ +0,68 pt) et celle de l'uplift observé (≈ +0,68 pt) coïncident, et sont égales à l'**ATE** de Mens (+0,68 pt). Le modèle ne sur- ni ne sous-estime l'effet global.

### Ce qui va dans le bon sens
- **Haut contre bas** : les 3 premiers quintiles (≈ +1,1 pt en moyenne) font nettement mieux que les 2 derniers (≈ 0 pt). Cela suggère qu'on pourrait écarter les ~40 % de clients les moins réceptifs sans perdre grand-chose.
- Dans les quintiles 4 et 5, le **taux du contrôle** (≈ 0,8 à 1,0 %) est plus élevé que dans les trois premiers (≈ 0,3 %) : le modèle isole des clients qui **convertissent déjà sans email** (profil « acquis »), pour lesquels l'envoi apporte peu.

### Ce qui pose problème
- **Non-monotonie** : le quintile 3 (+1,55 pt) dépasse le quintile 2 (+0,47 pt) et même le quintile 1. Un bon classement donnerait une décroissance régulière.
- **Écart prédit/observé** important par quintile (par exemple +0,67 prédit contre +1,55 observé au quintile 3).
- **Bruit très élevé** : environ 600 clients par groupe et par quintile, soit ≈ 10 conversions chez les traités et ≈ 2 chez le contrôle. L'erreur standard de chaque uplift observé est d'environ **0,6 pt**, soit un IC à 95 % de **±1,1 pt**. Presque tous les quintiles ont donc un IC qui contient 0, et les écarts entre quintiles sont **dans le bruit**.
- Le contraste haut contre bas (≈ 1,5 pt) correspond à un z d'environ 1,9 : **à la limite** de la significativité, sans correction.

### Ce que ça implique
- Le modèle capte **peut-être** un signal d'hétérogénéité modeste, cohérent avec la section 4 et avec l'uplift@30 positif, mais la preuve statistique est faible.
- Un ciblage « top 60 % » est **plausible**, avec un gain à valider sur le test, car le gain réel dépend du coût d'envoi (le gain d'une exclusion de 40 % est faible en valeur absolue).

### Limites et suite
- Ajouter des **IC par bootstrap** sur l'uplift observé de chaque quintile (barres d'erreur sur le graphique).
- Tester la **tendance** (régression de l'uplift observé sur le rang du quintile) plutôt que des écarts isolés.
- Comparer avec **3 quintiles** (moins bruités) ou agréger les quintiles 1 à 3 contre 4 et 5.
- **Valider sur le jeu de test** et reproduire pour **Womens**.
- Convertir le gain en **valeur** (conversions incrémentales × marge − coût d'envoi) pour choisir le seuil de ciblage.

## 5. Le prototype sert de test : l'implémentation du package doit coïncider

In [7]:
p = dict(_PARAMS)
a1 = HistGradientBoostingClassifier(random_state=cfg["seed"], **p).fit(Xf[mf][tf], yf[tf])
a0 = HistGradientBoostingClassifier(random_state=cfg["seed"], **p).fit(Xf[mf][~tf], yf[~tf])
proto = a1.predict_proba(Xv)[:, 1] - a0.predict_proba(Xv)[:, 1]
pkg = TLearner(cfg["seed"]).fit(Xf[mf], tf.astype(int), yf).predict_uplift(Xv)
assert np.allclose(proto, pkg), "le package diverge du prototype"
print("[X] TLearner du package == prototype notebook")

[X] TLearner du package == prototype notebook


**Conclusion** : le `TLearner` du package produit **exactement les mêmes uplifts** que le prototype écrit à la main dans le notebook. L'implémentation du package est donc fidèle à la logique validée dans les sections précédentes.

### Ce que le test vérifie
- Le prototype entraîne deux `HistGradientBoostingClassifier` (traités et contrôle), avec les mêmes hyperparamètres (`_PARAMS`) et la même graine, puis calcule $\hat p_1(x) - \hat p_0(x)$.
- Le package fait le même calcul via `TLearner(seed).fit(...).predict_uplift(...)`.
- `np.allclose` confirme que les deux vecteurs d'uplift sont égaux (à la tolérance numérique près) sur tout le jeu de validation. L'`assert` n'a pas levé d'erreur et le message final confirme la concordance.

### Pourquoi c'est utile
- Le notebook sert de **référence lisible** (« from scratch »), le package de **version réutilisable** : le test garantit qu'ils ne divergent pas.
- Il vérifie aussi le **traitement des types** : le notebook passe un masque booléen, le package reçoit `tf.astype(int)` et doit le convertir en booléen en interne. Une erreur à cet endroit (par exemple `~` appliqué à des entiers) donnerait des lignes fausses **sans message d'erreur** : ici, le résultat est identique, donc la conversion est correcte.
- Il protège contre les régressions : si on modifie le package plus tard (paramètres, ordre des colonnes, graine), le test casse.

### Limites
- **Les deux implémentations sont écrites par la même personne** avec la même logique : le test détecte les **erreurs de code**, pas les **erreurs conceptuelles** (par exemple une mauvaise définition de l'uplift).
- Il ne couvre que le **T-learner**, sur **un seul bras** et **une seule graine**. S-learner, X-learner, modèle de réponse et `MultiArmUplift` ne sont pas testés.
- Il utilise les **données réelles**, ce qui le rend dépendant du jeu de données et peu isolé.
- La tolérance par défaut de `np.allclose` (relative 1e-5, absolue 1e-8) est adaptée ici, mais elle reste à choisir en connaissance de cause pour des uplifts de l'ordre de 0,005.

### Suite proposée
- Déplacer ce test dans une suite **pytest** (`tests/`), avec des **données synthétiques** à effet connu : par exemple, un effet du traitement fixé à +5 pts doit être retrouvé par chaque learner.
- Ajouter des tests pour les autres learners et pour `MultiArmUplift`, plus des cas limites (un seul groupe dans les données, classe absente, entrées `DataFrame` ou NumPy).
- Tester la reproductibilité : deux appels avec la même graine donnent le même résultat, deux graines différentes peuvent différer.

## Décision

- **Candidats** : modèle de réponse et T-learner. Le S-learner (négatif sur Womens) et le X-learner (aucun gain) sont écartés.
- **Hyperparamètres** : arbres très peu profonds (`max_depth = 2`, `min_samples_leaf = 100`), la sensibilité étant faible et dominée par la profondeur.
- **Sélection** : automatique sur la **validation** dans `train.py`, selon le **Qini moyen** sur les deux bras. Les écarts entre candidats étant du même ordre que le bruit, ce choix reste fragile : il sera confirmé par bootstrap.
- **Évaluation finale** : **une seule fois** sur le jeu de test, sans réglage ultérieur.
- **Cohérence** : le `TLearner` du package reproduit le prototype du notebook (test d'égalité validé).
- **Suite** : `03_optimization_prototype.ipynb` (allocation du budget et choix du bras par client, en tenant compte du coût d'envoi).